# Estimación de la tasa de producción de petróleo a partir de parámetros operativos — Campo Volve

**Proyecto integrador · Cursos preparatorios · Maestría en Inteligencia Artificial Aplicada (UIDE)**
**Autor:** Alan Israel López Males  
**Fecha:** 15 de octubre de 2026

---
## 1. Planteamiento del problema

En ingeniería de producción, conocer la tasa de petróleo de cada pozo es clave para tomar decisiones operativas. Medirla de forma continua es costoso y los datos de sensores pueden fallar. Una alternativa es **estimarla a partir de variables operativas** (presiones, temperatura, apertura del choke), idea conocida como *medición virtual de flujo*.

**Pregunta:** ¿se puede estimar la tasa diaria de petróleo (`BORE_OIL_VOL`, en BPPD) de los pozos productores de Volve a partir de parámetros operativos, y qué tan bien generaliza el modelo a fechas posteriores?

**Enfoque (resumen):** un primer intento de predecir el petróleo directamente fracasa porque en Volve el corte de agua crece con el tiempo (hasta ~95 %). Por eso se plantea un modelo **en dos etapas**: (1) un modelo de ML estima la **tasa de líquido total** (petróleo + agua) a partir de presiones, temperatura y choke; (2) el petróleo se obtiene aplicando el **corte de agua de referencia** del pozo (mediana de sus 14 registros de producción previos).

**Alcance:** este trabajo estima la tasa **dadas unas condiciones operativas**; no es un pronóstico de declinación (curvas de Arps) ni un diagnóstico de causas.

**Objetivos**
1. Limpiar y analizar los datos de producción diaria de Volve.
2. Entrenar y comparar modelos de regresión con una partición **temporal** (entrenar con el pasado, evaluar con el futuro).
3. Interpretar los resultados e identificar qué variables pesan más.
4. Construir un detector simple de desviaciones y discutir sus límites.
5. Reflexionar sobre ética y limitaciones del uso de IA en este contexto.

## 2. Datos y atribución

- **Fuente:** conjunto de datos abierto del campo **Volve**, publicado por **Equinor** (archivo `Volve production data.xlsx`, hoja *Daily Production Data*).
- **Licencia:** Equinor Open Data Licence. Los datos no pueden revenderse y deben atribuirse a Equinor y a los antiguos socios de la licencia Volve. Equinor no ofrece garantías sobre su contenido ni relevancia.
- **Campo:** Volve, Mar del Norte; produjo entre 2008 y 2016.

## 2.1 Unidades de medida
El archivo original usa unidades del Sistema Internacional (Sm³, bar, °C). Como el proyecto se presenta en Ecuador, **todos los análisis, gráficos y resultados se expresan en unidades de campo**:

| Magnitud | Unidad original | Unidad de campo | Conversión |
|---|---|---|---|
| Tasa de petróleo | Sm³/día | **BPPD** (barriles de petróleo por día) | × 6,28981 |
| Tasa de agua | Sm³/día | **BAPD** (barriles de agua por día) | × 6,28981 |
| Tasa de líquido (petróleo + agua) | Sm³/día | **BFPD** (barriles de fluido por día) | × 6,28981 |
| Presiones (cabeza, fondo, diferenciales) | bar | **psi** | × 14,5038 |
| Temperaturas | °C | **°F** | × 9/5 + 32 |
| Apertura del choke | % | % (sin cambio) | — |
| Horas en línea | h | h (sin cambio) | — |

Este dataset no incluye variables de longitud (pies, pulgadas). Los barriles son barriles estándar (STB).

In [1]:
import os, sqlite3, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")

# --- Conversión de unidades SI -> unidades de campo ---
BBL_POR_SM3 = 6.28981     # 1 Sm3 de líquido = 6,28981 bbl (STB)
PSI_POR_BAR = 14.5038     # 1 bar = 14,5038 psi
def c_a_f(c):             # °C -> °F
    return c * 9 / 5 + 32

def a_unidades_campo(d):
    # Convierte volúmenes a bbl, presiones a psi y temperaturas a °F
    d = d.copy()
    for col in ["BORE_OIL_VOL", "BORE_WAT_VOL"]:
        d[col] = d[col] * BBL_POR_SM3
    for col in ["AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "AVG_ANNULUS_PRESS", "AVG_WHP_P", "DP_CHOKE_SIZE"]:
        d[col] = d[col] * PSI_POR_BAR
    for col in ["AVG_DOWNHOLE_TEMPERATURE", "AVG_WHT_P"]:
        d[col] = c_a_f(d[col])
    return d
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
SEMILLA = 42
np.random.seed(SEMILLA)

## 3. Carga de datos
El notebook busca el archivo en la carpeta `data/` del repositorio. Si se ejecuta en Google Colab sin el archivo, permite subirlo manualmente.

In [2]:
CANDIDATAS = [
    "data/Volve_production_data.xlsx",
    "data/Volve production data.xlsx",
    "Volve_production_data.xlsx",
    "Volve production data.xlsx",
    "/content/Volve_production_data.xlsx",
]
ruta = next((p for p in CANDIDATAS if os.path.exists(p)), None)
if ruta is None:
    try:
        from google.colab import files
        subido = files.upload()
        ruta = list(subido.keys())[0]
    except ImportError:
        raise FileNotFoundError("Coloca el archivo en la carpeta data/ del repositorio.")

print("Archivo:", ruta)
print("Hojas:", pd.ExcelFile(ruta).sheet_names)
raw = pd.read_excel(ruta, sheet_name="Daily Production Data")
print("Dimensiones (filas, columnas):", raw.shape)
raw.head()

Saving repositorio_proyecto_integrador.zip to repositorio_proyecto_integrador.zip
Archivo: repositorio_proyecto_integrador.zip


OptionError: No such keys(s): 'io.excel.zip.reader'

In [ ]:
print("Nota: el archivo original está en unidades SI (Sm3, bar, °C); aquí el petróleo acumulado se muestra en barriles.")
print("Rango de fechas:", raw["DATEPRD"].min().date(), "->", raw["DATEPRD"].max().date())
resumen = (raw.groupby(["NPD_WELL_BORE_NAME", "WELL_TYPE"])
              .agg(dias=("DATEPRD", "size"),
                   inicio=("DATEPRD", "min"),
                   fin=("DATEPRD", "max"),
                   petroleo_acumulado_bbl=("BORE_OIL_VOL", lambda x: x.sum() * BBL_POR_SM3))
              .round(0))
resumen

## 4. Consultas SQL sobre el dataset
Como parte de la exploración inicial se carga la tabla en una base SQLite en memoria y se consulta con SQL (conocimientos del curso de bases de datos).

In [ ]:
conn = sqlite3.connect(":memory:")
tmp = a_unidades_campo(raw)          # SQL sobre datos ya en unidades de campo
tmp["DATEPRD"] = tmp["DATEPRD"].dt.strftime("%Y-%m-%d")
tmp.to_sql("produccion", conn, index=False, if_exists="replace")

q1 = '''
SELECT NPD_WELL_BORE_NAME AS pozo,
       COUNT(*)                         AS dias_activos,
       ROUND(SUM(BORE_OIL_VOL), 0)      AS petroleo_total_bbl,
       ROUND(AVG(BORE_OIL_VOL), 1)      AS petroleo_prom_BPPD,
       ROUND(AVG(AVG_WHP_P), 1)         AS whp_prom_psi
FROM produccion
WHERE WELL_TYPE = 'OP' AND ON_STREAM_HRS > 0 AND BORE_OIL_VOL > 0
GROUP BY NPD_WELL_BORE_NAME
ORDER BY petroleo_total_bbl DESC
'''
display(pd.read_sql(q1, conn))

q2 = '''
SELECT SUBSTR(DATEPRD, 1, 4) AS anio,
       ROUND(SUM(BORE_OIL_VOL), 0) AS petroleo_bbl,
       ROUND(SUM(BORE_WAT_VOL), 0) AS agua_bbl
FROM produccion
WHERE WELL_TYPE = 'OP' AND ON_STREAM_HRS > 0
GROUP BY anio
ORDER BY anio
'''
display(pd.read_sql(q2, conn))

## 5. Limpieza de datos

Hallazgos de la inspección inicial y decisiones tomadas:

1. Se conservan solo **pozos productores** (`WELL_TYPE == "OP"`); los inyectores no tienen petróleo.
2. Se descartan los días **sin producción** (0 horas en línea o petróleo = 0): el modelo busca relacionar condiciones operativas con una tasa efectiva.
3. Los **ceros en presión y temperatura de fondo** se interpretan como sensor sin lectura y se convierten en datos faltantes.
4. La **presión anular** tiene demasiados faltantes y ceros, por lo que se descarta.
5. Se eliminan columnas constantes o administrativas.
6. Se calculan el **líquido total** (petróleo + agua) y el **corte de agua de referencia**, que se usará en el enfoque en dos etapas (ver sección 8).

In [ ]:
trazas = {}
trazas["Filas originales"] = len(raw)

df = raw[raw["WELL_TYPE"] == "OP"].copy()
trazas["Solo pozos productores (OP)"] = len(df)

df = df[(df["ON_STREAM_HRS"] > 0) & (df["BORE_OIL_VOL"] > 0)]
trazas["Con horas en línea y petróleo > 0"] = len(df)

# Ceros en sensores de fondo -> datos faltantes
sensores_fondo = ["AVG_DOWNHOLE_PRESSURE", "AVG_DOWNHOLE_TEMPERATURE"]
ceros_fondo = (df[sensores_fondo] == 0).sum()
print("Ceros en sensores de fondo:\n", ceros_fondo, "\n")
df[sensores_fondo] = df[sensores_fondo].replace(0, np.nan)

# Agua negativa (solo se usará para diagnóstico, no como variable de entrada)
print("Valores negativos de agua:", int((df["BORE_WAT_VOL"] < 0).sum()))
df["BORE_WAT_VOL"] = df["BORE_WAT_VOL"].clip(lower=0)

# Conversión a unidades de campo (después de tratar los ceros de sensor, porque 0 °C ≠ 0 °F)
df = a_unidades_campo(df)

SUPERFICIE = ["ON_STREAM_HRS", "AVG_CHOKE_SIZE_P", "AVG_WHP_P", "AVG_WHT_P", "DP_CHOKE_SIZE"]
FONDO = ["AVG_DOWNHOLE_PRESSURE", "AVG_DOWNHOLE_TEMPERATURE", "AVG_DP_TUBING"]
OBJETIVO = "BORE_OIL_VOL"
UNI = {"ON_STREAM_HRS": "h", "AVG_CHOKE_SIZE_P": "%", "AVG_WHP_P": "psi", "AVG_WHT_P": "°F", "DP_CHOKE_SIZE": "psi",
       "AVG_DOWNHOLE_PRESSURE": "psi", "AVG_DOWNHOLE_TEMPERATURE": "°F", "AVG_DP_TUBING": "psi", "BORE_OIL_VOL": "BPPD"}

df = df.dropna(subset=SUPERFICIE)   # variables de superficie esenciales
trazas["Sin faltantes en variables de superficie"] = len(df)

data = (df[["DATEPRD", "NPD_WELL_BORE_NAME"] + SUPERFICIE + FONDO + [OBJETIVO, "BORE_WAT_VOL"]]
          .rename(columns={"NPD_WELL_BORE_NAME": "POZO"})
          .sort_values(["POZO", "DATEPRD"]).reset_index(drop=True))

# Variables derivadas: líquido total, corte de agua y corte de agua de referencia
data["LIQUIDO"] = data[OBJETIVO] + data["BORE_WAT_VOL"]
data["CORTE_AGUA"] = data["BORE_WAT_VOL"] / data["LIQUIDO"]
# Referencia: mediana de los 14 registros previos del mismo pozo (sin incluir el día evaluado)
data["CORTE_AGUA_REF"] = data.groupby("POZO")["CORTE_AGUA"].transform(
    lambda x: x.rolling(14, min_periods=5).median().shift(1))
data = data.dropna(subset=["CORTE_AGUA_REF"]).sort_values("DATEPRD").reset_index(drop=True)
trazas["Con corte de agua de referencia (>=5 registros previos)"] = len(data)

print(pd.Series(trazas, name="filas").to_frame())
print("\nFaltantes por columna (%):")
print((data.isna().mean() * 100).round(1))

In [ ]:
# Disponibilidad de sensores de fondo por pozo
disp = data.groupby("POZO")[["AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING"]].apply(lambda g: g.notna().mean() * 100).round(1)
disp.columns = ["% con presion de fondo", "% con DP tubing"]
disp

## 6. Análisis exploratorio (EDA)

In [ ]:
vars_hist = SUPERFICIE + ["AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", OBJETIVO]
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, v in zip(axes.ravel(), vars_hist):
    sns.histplot(data[v].dropna(), bins=40, ax=ax, color="#3b6ea5")
    ax.set_title(f"{v} ({UNI[v]})"); ax.set_xlabel("")
plt.suptitle("Distribución de las variables", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
sns.scatterplot(data=data, x="DATEPRD", y=OBJETIVO, hue="POZO", s=8, linewidth=0, ax=ax)
ax.set_ylabel("Petróleo (BPPD)"); ax.set_xlabel("Fecha")
ax.set_title("Tasa diaria de petróleo por pozo")
ax.legend(markerscale=2, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, v in zip(axes, ["AVG_WHP_P", "DP_CHOKE_SIZE"]):
    sns.scatterplot(data=data, x=v, y=OBJETIVO, hue="POZO", s=8, linewidth=0, ax=ax, legend=(v == "DP_CHOKE_SIZE"))
    ax.set_title(f"{OBJETIVO} (BPPD) vs {v} ({UNI[v]})")
axes[1].legend(markerscale=2, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
corr = data[SUPERFICIE + FONDO + [OBJETIVO]].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True, cbar_kws={"shrink": .8})
plt.title("Matriz de correlación"); plt.tight_layout(); plt.show()
print(corr[OBJETIVO].drop(OBJETIVO).sort_values(ascending=False).round(2))

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(data=data, x="POZO", y=OBJETIVO, color="#9db8d6")
plt.title("Distribución de la tasa de petróleo por pozo"); plt.ylabel("BPPD")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
anual = data.groupby(data["DATEPRD"].dt.year)[[OBJETIVO, "LIQUIDO"]].mean()
anual.plot(ax=axes[0], marker="o")
axes[0].set_title("Tasa media anual: petróleo vs líquido total"); axes[0].set_ylabel("bbl/d"); axes[0].set_xlabel("Año")
sns.lineplot(data=data.assign(ANIO=data["DATEPRD"].dt.year), x="ANIO", y="CORTE_AGUA", hue="POZO", marker="o", ax=axes[1], errorbar=None)
axes[1].set_title("Corte de agua medio anual por pozo"); axes[1].set_xlabel("Año")
axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.show()

**Observaciones del EDA** _(complétalas con tus propias palabras al revisar los gráficos: relaciones entre variables, diferencias entre pozos, comportamiento en el tiempo, valores atípicos)._

## 7. Preparación y partición temporal

Los datos son una serie de tiempo, por lo que una partición aleatoria mezclaría días vecinos entre entrenamiento y prueba y daría resultados optimistas. Se **entrena con el período anterior a julio de 2015** y se **evalúa con el posterior**. El período de prueba incluye un pozo (F-5) que casi no aparece en el entrenamiento, lo que pone a prueba la generalización.

**Fuga de datos:** no se usan `BORE_GAS_VOL` ni `BORE_WAT_VOL` como entradas, porque son resultados de la producción del mismo día. El agua se usará solo en la etapa de diagnóstico.

In [ ]:
CORTE = pd.Timestamp("2015-07-01")
train = data[data["DATEPRD"] < CORTE]
test = data[data["DATEPRD"] >= CORTE]
print(f"Entrenamiento: {len(train)} filas ({train.DATEPRD.min().date()} a {train.DATEPRD.max().date()})")
print(f"Prueba:        {len(test)} filas ({test.DATEPRD.min().date()} a {test.DATEPRD.max().date()})")
print("\nPozos en prueba y filas en entrenamiento:")
print(pd.DataFrame({"prueba": test.POZO.value_counts(), "entrenamiento": train.POZO.value_counts()}).fillna(0).astype(int))

## 8. Modelos

### 8.1 Intento 1: predecir el petróleo directamente
Primero se entrena a los modelos para predecir `BORE_OIL_VOL` directamente con las variables operativas de superficie. Se comparan un **dummy** (predice la media de entrenamiento, referencia mínima), una **regresión lineal**, un **Random Forest** y una **red neuronal** pequeña (MLP).

In [ ]:
def metricas(y, p):
    return {"R2": r2_score(y, p),
            "RMSE": np.sqrt(mean_squared_error(y, p)),
            "MAE": mean_absolute_error(y, p)}

def crear_modelos():
    return {
        "Dummy (media)": Pipeline([("imp", SimpleImputer(strategy="median")), ("m", DummyRegressor())]),
        "Regresión lineal": Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("m", LinearRegression())]),
        "Random Forest": Pipeline([("imp", SimpleImputer(strategy="median")),
                                   ("m", RandomForestRegressor(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEMILLA))]),
        # La red neuronal estandariza también el objetivo (así el resultado no depende de las unidades: bbl o Sm3)
        "Red neuronal (MLP)": TransformedTargetRegressor(
            regressor=Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                                ("m", MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=800, early_stopping=True, random_state=SEMILLA))]),
            transformer=StandardScaler()),
    }

res1 = []
for nombre, modelo in crear_modelos().items():
    modelo.fit(train[SUPERFICIE], train[OBJETIVO])
    fila = {"Modelo": nombre}
    fila.update(metricas(test[OBJETIVO], modelo.predict(test[SUPERFICIE])))
    res1.append(fila)
pd.DataFrame(res1).sort_values("RMSE").round(3)

**Lectura del Intento 1:** los R² son malos o negativos incluso con el modelo más flexible. Hay una razón física que se ve en las gráficas del EDA: el petróleo producido depende del **corte de agua**, que en Volve sube con el tiempo (en el período de prueba la tasa media de petróleo es mucho menor que en el de entrenamiento, aunque el pozo siga produciendo líquido). Las variables operativas (presiones y choke) explican cuánto **líquido** sale del pozo, pero no cuánto de ese líquido es petróleo. Por eso el modelo, entrenado con un período de mucho petróleo, sobreestima en el período de prueba.

### 8.2 Enfoque en dos etapas
1. **Etapa 1 (ML):** estimar la **tasa de líquido total** a partir de las variables operativas.
2. **Etapa 2 (física):** `petróleo estimado = líquido estimado × (1 − corte de agua de referencia)`, usando la mediana del corte de agua de los 14 registros previos del pozo. En campo, el corte de agua se conoce por pruebas de pozo o asignación de producción, y cambia lentamente.

Se comparan dos conjuntos de variables:
- **Superficie:** horas en línea, choke, presión y temperatura en cabeza, diferencial en el choke.
- **Superficie + fondo:** añade presión y temperatura de fondo y diferencial en tubería (con ~26 % de datos faltantes, que se imputan con la mediana; el pozo F-5 no tiene datos de fondo).

Como referencia de **límite superior**, se calcula también qué pasaría con un estimador de líquido perfecto (líquido real × (1 − corte de agua de referencia)): mide el error que introduce solo la suposición del corte de agua.

In [ ]:
conjuntos = {"Superficie": SUPERFICIE, "Superficie + fondo": SUPERFICIE + FONDO}
modelos_ml = {k: v for k, v in crear_modelos().items() if k != "Dummy (media)"}
resultados, ajustados = [], {}

for nombre_set, feats in conjuntos.items():
    for nombre_mod, modelo in crear_modelos().items():
        modelo.fit(train[feats], train["LIQUIDO"])
        liq_pred = np.clip(modelo.predict(test[feats]), 0, None)
        petroleo_pred = liq_pred * (1 - test["CORTE_AGUA_REF"])
        fila = {"Variables": nombre_set, "Modelo": nombre_mod,
                "R2 líquido": r2_score(test["LIQUIDO"], liq_pred)}
        m = metricas(test[OBJETIVO], petroleo_pred)
        fila.update({"R2 petróleo": m["R2"], "RMSE petróleo": m["RMSE"], "MAE petróleo": m["MAE"]})
        resultados.append(fila)
        ajustados[(nombre_set, nombre_mod)] = modelo

tabla = pd.DataFrame(resultados).sort_values("RMSE petróleo").reset_index(drop=True)
display(tabla.round(3))

limite = metricas(test[OBJETIVO], test["LIQUIDO"] * (1 - test["CORTE_AGUA_REF"]))
print("Límite superior (líquido real × (1 − corte de agua de referencia)):", {k: round(v, 3) for k, v in limite.items()})

In [ ]:
# Sensibilidad de la red neuronal a la semilla aleatoria (con las variables de superficie)
filas = []
for sem in range(5):
    mlp = crear_modelos()["Red neuronal (MLP)"]
    mlp.set_params(regressor__m__random_state=sem)
    mlp.fit(train[SUPERFICIE], train["LIQUIDO"])
    p = np.clip(mlp.predict(test[SUPERFICIE]), 0, None) * (1 - test["CORTE_AGUA_REF"])
    filas.append(metricas(test[OBJETIVO], p))
sens = pd.DataFrame(filas)
print(sens.round(3))
print(f"\nRMSE medio ± desv: {sens.RMSE.mean():.1f} ± {sens.RMSE.std():.1f}")

**Lectura de resultados:** compara los modelos de la tabla (¿mejora el Random Forest a la regresión lineal? ¿ayudan las variables de fondo o empeoran por los datos faltantes?) y observa cuánto varía la red neuronal al cambiar la semilla (celda anterior) frente a un modelo de árboles como el Random Forest. _Complétala con tus propias palabras._

## 9. Evaluación del modelo final (Random Forest)

In [ ]:
# Criterio de selección: se usa el Random Forest con variables de superficie como modelo final por su estabilidad
# (resultado reproducible) y porque permite interpretar la importancia de variables. La red neuronal obtuvo el menor
# RMSE con la semilla 42, pero su desempeño varía mucho con la semilla (ver la sensibilidad más arriba).
set_mejor, mod_mejor = "Superficie", "Random Forest"
print("Modelo final:", mod_mejor, "| variables:", set_mejor)
print("Mejor RMSE de petróleo en la tabla de prueba:", tabla.iloc[0]["Modelo"], "-", tabla.iloc[0]["Variables"])
feats_mejor = conjuntos[set_mejor]
modelo_final = ajustados[(set_mejor, mod_mejor)]

test = test.copy()
test["PRED_LIQ"] = np.clip(modelo_final.predict(test[feats_mejor]), 0, None)
test["PRED"] = test["PRED_LIQ"] * (1 - test["CORTE_AGUA_REF"])
test["RESID"] = test[OBJETIVO] - test["PRED"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=test, x=OBJETIVO, y="PRED", hue="POZO", s=12, linewidth=0, ax=axes[0])
lim = [0, max(test[OBJETIVO].max(), test["PRED"].max()) * 1.05]
axes[0].plot(lim, lim, "k--", lw=1); axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel("Real (BPPD)"); axes[0].set_ylabel("Predicho (BPPD)"); axes[0].set_title("Petróleo: real vs predicho (período de prueba)")
sns.histplot(test["RESID"], bins=50, ax=axes[1], color="#c0504d")
axes[1].set_title("Distribución de residuos (real - predicho)"); axes[1].set_xlabel("BPPD")
plt.tight_layout(); plt.show()

In [ ]:
pozos_test = test["POZO"].unique()
fig, axes = plt.subplots(len(pozos_test), 1, figsize=(12, 2.6 * len(pozos_test)), sharex=True)
for ax, p in zip(np.atleast_1d(axes), pozos_test):
    s_ = test[test["POZO"] == p].sort_values("DATEPRD")
    ax.plot(s_["DATEPRD"], s_[OBJETIVO], label="Real", lw=1)
    ax.plot(s_["DATEPRD"], s_["PRED"], label="Predicho", lw=1)
    ax.set_title(p, loc="left", fontsize=10); ax.set_ylabel("BPPD")
np.atleast_1d(axes)[0].legend(ncol=2)
plt.tight_layout(); plt.show()

print("Desempeño por pozo en el período de prueba (petróleo):")
por_pozo = test.groupby("POZO").apply(lambda g: pd.Series({
    "n": len(g),
    "RMSE": np.sqrt(mean_squared_error(g[OBJETIVO], g["PRED"])),
    "MAE": mean_absolute_error(g[OBJETIVO], g["PRED"]),
    "R2": r2_score(g[OBJETIVO], g["PRED"]) if len(g) > 2 else np.nan}))
por_pozo.round(2)

## 10. Interpretación: ¿qué variables pesan más?
Se usa **importancia por permutación** sobre el conjunto de prueba, aplicada al modelo de líquido (etapa 1): se mezcla una variable y se mide cuánto empeora el RMSE del líquido. Es más confiable que la importancia interna de los árboles cuando hay variables correlacionadas, aunque en ese caso puede repartir el efecto entre ellas.

In [ ]:
imp = permutation_importance(modelo_final, test[feats_mejor], test["LIQUIDO"],
                             n_repeats=10, random_state=SEMILLA, scoring="neg_root_mean_squared_error")
imp_df = pd.DataFrame({"variable": feats_mejor, "aumento_RMSE": imp.importances_mean, "desv": imp.importances_std}) \
           .sort_values("aumento_RMSE", ascending=True)
plt.figure(figsize=(8, 4.5))
plt.barh(imp_df["variable"], imp_df["aumento_RMSE"], xerr=imp_df["desv"], color="#3b6ea5")
plt.xlabel("Aumento del RMSE del líquido al permutar la variable (BFPD)")
plt.title("Importancia por permutación (etapa 1: líquido)"); plt.tight_layout(); plt.show()
imp_df.sort_values("aumento_RMSE", ascending=False).round(2)

**Interpretación** _(escribe con tus palabras: qué variables dominan y si tiene sentido físico, por ejemplo la relación entre presión en cabeza, apertura del choke y caída de presión por el choke; qué pozos se predicen mejor o peor y posibles razones, como la falta de historia de F-5 en el entrenamiento)._

## 11. Extensión: detector de desviaciones y orientación por reglas

Cuando la producción real se aleja de lo que el modelo espera para esas condiciones operativas, puede haber un cambio no explicado por las variables medidas. Se marca como **desviación** un día con residuo mayor a 3 desviaciones robustas (basadas en la MAD).

Sobre esos días se aplican **reglas de ingeniería simples** que sugieren **hipótesis** para que el ingeniero las verifique. **No son un diagnóstico**: el dataset no tiene eventos etiquetados (emulsión, fugas, recirculación), así que el sistema no puede confirmar causas.

In [ ]:
sigma_rob = 1.4826 * np.median(np.abs(test["RESID"] - np.median(test["RESID"])))
UMBRAL = 3 * sigma_rob
test["ANOMALIA"] = test["RESID"].abs() > UMBRAL
print(f"Desviación robusta: {sigma_rob:.1f} BPPD  ->  umbral: ±{UMBRAL:.1f} BPPD")
print(f"Días marcados: {int(test['ANOMALIA'].sum())} de {len(test)} ({test['ANOMALIA'].mean()*100:.1f} %)")

p_demo = test["POZO"].value_counts().index[0]
s = test[test["POZO"] == p_demo].sort_values("DATEPRD")
plt.figure(figsize=(12, 4))
plt.plot(s["DATEPRD"], s[OBJETIVO], lw=1, label="Real")
plt.plot(s["DATEPRD"], s["PRED"], lw=1, label="Esperado por el modelo")
a = s[s["ANOMALIA"]]
plt.scatter(a["DATEPRD"], a[OBJETIVO], color="red", s=18, zorder=3, label="Desviación")
plt.title(f"Pozo {p_demo}: desviaciones detectadas"); plt.ylabel("BPPD"); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Referencia móvil por pozo (14 registros previos, sin incluir el día evaluado) para las demás variables.
# El corte de agua del día se usa solo para diagnóstico, no como entrada del modelo.
ref = data.sort_values(["POZO", "DATEPRD"]).copy()
for v in ["AVG_WHP_P", "AVG_CHOKE_SIZE_P", "DP_CHOKE_SIZE"]:
    ref[v + "_REF"] = ref.groupby("POZO")[v].transform(lambda x: x.rolling(14, min_periods=5).median().shift(1))
test = test.merge(ref[["DATEPRD", "POZO"] + [c for c in ref.columns if c.endswith("_REF") and c != "CORTE_AGUA_REF"]],
                  on=["DATEPRD", "POZO"], how="left")

def hipotesis(r):
    if not r["ANOMALIA"]:
        return ""
    choke_estable = abs(r["AVG_CHOKE_SIZE_P"] - r["AVG_CHOKE_SIZE_P_REF"]) <= 5      # puntos porcentuales
    d_whp = (r["AVG_WHP_P"] - r["AVG_WHP_P_REF"]) / r["AVG_WHP_P_REF"] if r["AVG_WHP_P_REF"] else np.nan
    d_dpc = (r["DP_CHOKE_SIZE"] - r["DP_CHOKE_SIZE_REF"]) / r["DP_CHOKE_SIZE_REF"] if r["DP_CHOKE_SIZE_REF"] else np.nan
    d_agua = r["CORTE_AGUA"] - r["CORTE_AGUA_REF"]   # diagnóstico: cambio frente a la referencia
    if r["RESID"] > 0:
        return "Producción mayor a la esperada: revisar medición / cambios recientes de choke o pozo"
    if choke_estable and pd.notna(d_agua) and d_agua > 0.05:
        return "Caída con aumento de corte de agua: revisar entrada de agua, conificación o emulsión (verificar)"
    if choke_estable and pd.notna(d_whp) and d_whp < -0.15:
        return "Caída de presión en cabeza con choke estable: revisar restricción aguas arriba o pérdida de integridad (verificar)"
    if choke_estable and pd.notna(d_dpc) and d_dpc > 0.30:
        return "Aumento del diferencial en el choke: posible restricción o taponamiento (verificar)"
    return "Sin hipótesis por reglas: revisar históricos del pozo"

test["HIPOTESIS"] = test.apply(hipotesis, axis=1)
resumen_hip = test.loc[test["ANOMALIA"], "HIPOTESIS"].value_counts().rename("dias").to_frame()
display(resumen_hip)
cols = ["DATEPRD", "POZO", OBJETIVO, "PRED", "RESID", "HIPOTESIS"]
test.loc[test["ANOMALIA"], cols].sort_values("RESID").head(8).round(1)

**Lectura crítica:** las hipótesis son solo orientativas. Los umbrales de las reglas (5 puntos de choke, 15 % en presión, 30 % en el diferencial, 5 puntos de corte de agua) se fijaron por criterio y deberían calibrarse con un especialista y con eventos reales. Además, el umbral de desviación se calculó con los mismos datos de prueba, por lo que para un uso real habría que calibrarlo con un período de validación independiente.

## 12. Guardado del modelo
Se guarda el modelo y la lista de variables para poder reutilizarlos después (por ejemplo, en una interfaz con Streamlit).

In [ ]:
os.makedirs("models", exist_ok=True)
# Para uso posterior, se reajusta el mejor modelo de líquido con todos los datos disponibles.
# Petróleo estimado = líquido estimado × (1 − corte de agua de referencia)
modelo_prod = crear_modelos()[mod_mejor]
modelo_prod.fit(data[feats_mejor], data["LIQUIDO"])
joblib.dump({"modelo_liquido": modelo_prod, "variables": feats_mejor, "umbral_desviacion": float(UMBRAL)},
            "models/modelo_produccion.joblib")
rangos = data[feats_mejor + ["CORTE_AGUA_REF"]].describe().loc[["min", "max", "50%"]].T.round(3)
rangos.to_json("models/rangos_variables.json")
print("Modelo guardado en models/modelo_produccion.joblib")
rangos

## 13. Ética, riesgos y limitaciones del uso de IA

- **Alcance de los datos:** son de un solo campo y de pozos concretos, así que el modelo no debe aplicarse a otros campos sin revalidar. Tampoco extrapola bien fuera del rango de condiciones vistas.
- **Supuesto del corte de agua:** la estimación de petróleo depende de conocer un corte de agua reciente. En campo viene de pruebas de pozo o asignación de producción y puede tener errores o retrasos; si el corte cambia de golpe (por ejemplo, irrupción de agua), la estimación quedará sesgada.
- **Calidad de datos y sesgos:** los ceros y faltantes en sensores, y los pozos con poca historia (como F-5), afectan el desempeño. Un buen R² global puede ocultar errores grandes en un pozo específico.
- **Uso responsable:** una estimación no reemplaza la medición real ni la validación de un ingeniero. Decisiones operativas, económicas o de seguridad no deberían basarse solo en este modelo.
- **Explicaciones y causas:** el modelo muestra asociaciones, no causalidad. Las hipótesis de la sección 11 son orientativas y podrían inducir a error si se toman como diagnóstico.
- **Transparencia y reproducibilidad:** se documentan decisiones de limpieza, partición y métricas, y se fijó la semilla aleatoria.
- **Licencia y atribución:** los datos pertenecen a Equinor y los socios de Volve, y se usan bajo la Equinor Open Data Licence, con fines de aprendizaje.
- **Impacto y supervisión:** si un sistema así se usara en producción, requeriría monitoreo continuo, reentrenamiento y un responsable humano que valide las alertas.

## 14. Conclusiones
_(Redáctalas con tus palabras al terminar de revisar los resultados. Sugerencia de contenido:)_
1. Qué modelo y qué variables funcionaron mejor, con las métricas de la sección 8.
2. Qué tan bien generaliza al período y a los pozos de prueba, y dónde falla.
3. Qué variables dominan y si coincide con lo esperado en ingeniería de producción.
4. Qué aporta y qué no el detector de desviaciones.
5. Qué mejorarías con más tiempo o datos (más pozos, eventos etiquetados, comparación con curvas de declinación, interfaz de usuario).

## Referencias
- Equinor. *Volve field data set* (Equinor Open Data Licence). https://www.equinor.com/energy/volve-data-sharing
- Documentación de scikit-learn, pandas, matplotlib y seaborn.